Exécuter ces commandes à partir de la machine hôte
```bash
# Run MySql container
docker run -it -d -p 3306:3306 -v ./mysqlsampledatabase.sql:/mysqlsampledatabase.sql --name mysql -e MYSQL_ROOT_PASSWORD=root mysql:8.0.27 
# Create Sample Database (classsicmodels)
docker exec mysql bash -c "mysql -u root --password=root </mysqlsampledatabase.sql"
# Get IP Address of MySQL Server
docker inspect --format='{{.NetworkSettings.IPAddress}}' mysql
```

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession 
    .builder 
    .appName("Read and write with Mysql") 
    .config('spark.jars.packages', 'com.mysql:mysql-connector-j:9.1.0')
    .config("spark.sql.shuffle.partitions", 8)
    .master("local[*]") 
    .getOrCreate()
)

In [3]:
jdbcDF = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:mysql://172.17.0.3:3306/") \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .option("dbtable", "classicmodels.orders") \
    .option("user", "root") \
    .option("password", "root") \
    .load()
jdbcDF.show()

+-----------+----------+------------+-----------+-------+--------------------+--------------+
|orderNumber| orderDate|requiredDate|shippedDate| status|            comments|customerNumber|
+-----------+----------+------------+-----------+-------+--------------------+--------------+
|      10100|2003-01-06|  2003-01-13| 2003-01-10|Shipped|                NULL|           363|
|      10101|2003-01-09|  2003-01-18| 2003-01-11|Shipped|Check on availabi...|           128|
|      10102|2003-01-10|  2003-01-18| 2003-01-14|Shipped|                NULL|           181|
|      10103|2003-01-29|  2003-02-07| 2003-02-02|Shipped|                NULL|           121|
|      10104|2003-01-31|  2003-02-09| 2003-02-01|Shipped|                NULL|           141|
|      10105|2003-02-11|  2003-02-21| 2003-02-12|Shipped|                NULL|           145|
|      10106|2003-02-17|  2003-02-24| 2003-02-21|Shipped|                NULL|           278|
|      10107|2003-02-24|  2003-03-03| 2003-02-26|Shipped|Dif

In [8]:
orderdetails_df = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:mysql://172.17.0.3:3306/") \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .option("dbtable", "classicmodels.orderdetails") \
    .option("user", "root") \
    .option("password", "root") \
    .load()

orderdetails_df.show()

+-----------+-----------+---------------+---------+---------------+
|orderNumber|productCode|quantityOrdered|priceEach|orderLineNumber|
+-----------+-----------+---------------+---------+---------------+
|      10100|   S18_1749|             30|   136.00|              3|
|      10100|   S18_2248|             50|    55.09|              2|
|      10100|   S18_4409|             22|    75.46|              4|
|      10100|   S24_3969|             49|    35.29|              1|
|      10101|   S18_2325|             25|   108.06|              4|
|      10101|   S18_2795|             26|   167.06|              1|
|      10101|   S24_1937|             45|    32.53|              3|
|      10101|   S24_2022|             46|    44.35|              2|
|      10102|   S18_1342|             39|    95.55|              2|
|      10102|   S18_1367|             41|    43.13|              1|
|      10103|   S10_1949|             26|   214.30|             11|
|      10103|   S10_4962|             42|   119.

In [9]:
import pyspark.sql.functions as F
joined_df = jdbcDF.join(orderdetails_df, "orderNumber")

prix_moyen_par_commande = joined_df.groupBy("orderNumber").agg(
    F.avg("priceEach").alias("prix_moyen_commande"),
    F.sum("quantityOrdered").alias("quantite_totale"),
    F.sum(F.col("priceEach") * F.col("quantityOrdered")).alias("montant_total")
)

prix_moyen_par_commande.show()

+-----------+-------------------+---------------+-------------+
|orderNumber|prix_moyen_commande|quantite_totale|montant_total|
+-----------+-------------------+---------------+-------------+
|      10103|          95.023125|            541|     50218.95|
|      10113|          81.307500|            143|     11044.30|
|      10148|          98.207143|            430|     41554.73|
|      10158|          67.790000|             22|      1491.38|
|      10170|         102.555000|            142|     15130.97|
|      10179|          78.093333|            288|     22963.60|
|      10192|          96.317500|            585|     55425.77|
|      10195|          91.146000|            385|     36092.40|
|      10197|          79.223571|            503|     40473.86|
|      10205|          69.490000|            180|     13059.16|
|      10206|          96.868182|            355|     36527.61|
|      10258|         104.080000|            200|     22037.91|
|      10263|          98.017273|       

In [10]:
prix_moyen_par_commande.write.saveAsTable("Results")

In [11]:
spark.sql("select * from Results").show()

+-----------+-------------------+---------------+-------------+
|orderNumber|prix_moyen_commande|quantite_totale|montant_total|
+-----------+-------------------+---------------+-------------+
|      10103|          95.023125|            541|     50218.95|
|      10113|          81.307500|            143|     11044.30|
|      10148|          98.207143|            430|     41554.73|
|      10158|          67.790000|             22|      1491.38|
|      10170|         102.555000|            142|     15130.97|
|      10179|          78.093333|            288|     22963.60|
|      10192|          96.317500|            585|     55425.77|
|      10195|          91.146000|            385|     36092.40|
|      10197|          79.223571|            503|     40473.86|
|      10205|          69.490000|            180|     13059.16|
|      10206|          96.868182|            355|     36527.61|
|      10258|         104.080000|            200|     22037.91|
|      10263|          98.017273|       

In [12]:
spark.sql("show tables").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|  default|  results|      false|
+---------+---------+-----------+

